# 28. Stream metrics, for comparison with the adapter plasticity study

The adapter plasticity study in `reference/adapter-plasticity-study` reports four summary
numbers plus a curve, all taken from one accuracy matrix per run. This notebook computes
the same five for this framework, so the two projects can be put side by side.

### The accuracy matrix

A(i, j) is the accuracy on task j after the stream has trained through task i. Only the
lower triangle exists, because a task has no accuracy before it is learned.

### The five numbers

| Name | What it is | Direction |
| --- | --- | --- |
| Final accuracy | mean over every task at the end | higher is better |
| **Learning accuracy** | mean of the diagonal A(t, t) | higher is better |
| Forgetting | mean drop from each task's best to its last | lower is better |
| Backward transfer | mean change from each task's first to its last | higher is better |
| **New-task accuracy curve** | the diagonal itself, summarised by its slope | falling means worse |

The two in bold are the plasticity ones. The adapter study calls learning accuracy "a
direct within-stream measure of plasticity", because it says how well each task was learned
at the moment it was learned.

### What this notebook costs

Nothing. It needs no GPU, trains nothing, and downloads only small JSON reports. Every
number comes from records the study already saved.

### Two differences from the adapter study, both unavoidable

**Only the task-incremental form exists here.** That study reports every metric twice,
class-incremental and task-incremental, because its model has one head over all classes.
This framework has no such head: a task code generates a ten-output classifier, and the
code must be supplied to get any prediction. The adapter study calls the task-incremental
form the cleaner read, so this is the comparable one, but it is not a free choice.

**Eight tasks, not twenty.** The slope is reported per task and also scaled to per ten
tasks, which is how the adapter study reports it. The scaled figure is for comparison only;
with eight points it is an extrapolation.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/path-stream-metrics")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
if importlib.util.find_spec("boto3") is None:
    if not IN_COLAB:
        raise ImportError("Install boto3 first.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3"], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle.stream_metrics import accuracy_matrix, across_seeds

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("No GPU needed. This notebook reads saved reports and computes.")

## 2. Settings

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

# Which learning order to analyse. order_01 is the one the study completed.
# Orders 02 and 03 were planned and deferred; see section 6 for why they matter.
ORDER = "order_01"

SAVE_TO_R2 = True
print("Study:", STUDY_ID, "| order:", ORDER)

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Build one accuracy matrix per seed

A source job learns the eight tasks of its order and forgets nothing, so its saved history
is exactly the learning stream these metrics are defined for. The matrix builder refuses a
history containing a forget request, because a removed task would otherwise be counted as
catastrophic forgetting.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
print("Learning order:", " then ".join(ORDER_TASKS))

matrices = {}
for seed in plan["seeds"]:
    job = next((j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{seed}"), None)
    report = storage.completed_report(job) if job else None
    if report is None:
        print(f"  seed {seed}: source job not finished, skipped")
        continue
    matrices[str(seed)] = accuracy_matrix(report["history"], ORDER_TASKS)
    print(f"  seed {seed}: matrix built from {len(report['history'])} learning requests")

if not matrices:
    raise RuntimeError("No finished source jobs for this order.")
result = across_seeds(matrices)

## 5. The results

In [ ]:
seeds = result["seeds"]
size = len(ORDER_TASKS)
WIDTH = 94

def rule(title):
    print(f"\n{title}\n" + "-" * WIDTH)

print("=" * WIDTH)
print("STREAM METRICS".center(WIDTH))
print(f"study {STUDY_ID} | {ORDER} | {len(seeds)} seeds | {size} tasks | "
      "task-incremental".center(WIDTH))
print("=" * WIDTH)

rule("TABLE 1. Accuracy matrix, percent. Rows are after task i, columns are task j")
for seed in seeds:
    print(f"\nseed {seed}")
    print(f"{'after':>8} " + " ".join(f"{('task ' + t):>8}" for t in ORDER_TASKS))
    for i, row in enumerate(result["per_seed"][seed] and matrices[seed]):
        cells = " ".join(f"{v:>8.1f}" if v is not None else f"{'':>8}" for v in row)
        print(f"{ORDER_TASKS[i]:>8} {cells}")

rule("TABLE 2. New-task accuracy curve, A(t,t). How well each task was learned")
print(f"{'seed':>6} " + " ".join(f"{('pos ' + str(i + 1)):>8}" for i in range(size))
      + f" {'slope':>9}")
print(f"{'':>6} " + " ".join(f"{('task ' + t):>8}" for t in ORDER_TASKS) + f" {'per task':>9}")
print("-" * WIDTH)
for seed in seeds:
    per = result["per_seed"][seed]
    print(f"{seed:>6} " + " ".join(f"{v:>8.1f}" for v in per["diagonal"])
          + f" {per['learning_curve_slope_per_task']:>+9.2f}")
mean_diagonal = [sum(result["per_seed"][s]["diagonal"][i] for s in seeds) / len(seeds)
                 for i in range(size)]
print("-" * WIDTH)
print(f"{'mean':>6} " + " ".join(f"{v:>8.1f}" for v in mean_diagonal))
print("\nA falling curve means each new task is learned less well than the one before.")

rule("TABLE 3. The four summary metrics, mean and sample SD over seeds")
print(f"{'metric':<34}{'mean':>9}{'sd':>8}{'95% CI over seeds':>22}  per seed")
print("-" * WIDTH)
labels = {
    "final_accuracy": "Final accuracy, pct",
    "learning_accuracy": "Learning accuracy, pct",
    "forgetting": "Forgetting, points",
    "backward_transfer": "Backward transfer, points",
    "learning_curve_slope_per_task": "Curve slope, pts per task",
    "learning_curve_slope_per_10_tasks": "Curve slope, pts per 10 tasks",
}
for key, label in labels.items():
    row = result["across_seeds"][key]
    interval = f"[{row['ci_low']:.2f}, {row['ci_high']:.2f}]"
    values = " ".join(f"{v:.2f}" for v in row["values"])
    print(f"{label:<34}{row['mean']:>9.2f}{row['sd']:>8.2f}{interval:>22}  {values}")
print("-" * WIDTH)
print(f"The interval is a 95% percentile bootstrap over {len(seeds)} seeds, 10,000 resamples,")
print("as the adapter study specifies. With this few seeds it is wide and its ends fall")
print("on a few distinct values, so read it with the seed count beside it.")

## 6. How to read the slope, and why it is not yet a finding

The adapter study declares plasticity loss only when **both** of two conditions hold:

1. the 95% paired bootstrap interval over seeds excludes zero in the loss direction, and
2. the mean change exceeds a noise floor estimated from repeated measurements at t = 0.

The cell below checks the first. **It cannot check the second**, because the stream runs
were never repeated within a seed, so there is no repetition spread to build a noise floor
from. One condition is not the rule.

There is a larger problem, and it is the one to say out loud first.

**Task identity and position are confounded.** Every seed here runs the same learning order,
so position 7 is always task 7 and position 8 is always task 14. If those two tasks happen
to be harder than the others, the curve falls for that reason alone and plasticity has
nothing to do with it. The adapter study avoids this by probing the *same* held-out tasks at
every point. The in-stream diagonal cannot, because each point is a different task.

The study plan already contains two further orders that would separate the two. Position 8
is task 14 in order 01, task 5 in order 02 and task 3 in order 03. Running those would show
whether the dip follows the position or follows the task.

In [ ]:
slope_row = result["across_seeds"]["learning_curve_slope_per_task"]
all_negative = all(v < 0 for v in slope_row["values"])
excludes_zero = slope_row["ci_high"] < 0

print(f"Slope per task, per seed: " + ", ".join(f"{v:+.2f}" for v in slope_row["values"]))
print(f"Mean {slope_row['mean']:+.2f}, "
      f"95% interval [{slope_row['ci_low']:.2f}, {slope_row['ci_high']:.2f}]")
print()
print(f"  Condition 1, interval excludes zero in the loss direction: "
      f"{'MET' if excludes_zero else 'not met'}")
print(f"  Condition 2, mean exceeds the noise floor: CANNOT BE CHECKED")
print("     The stream runs were not repeated within a seed, so there is no")
print("     repetition spread to estimate a noise floor from.")
print()
print("VERDICT: not a finding. The decision rule needs both conditions, and the")
print("second cannot be evaluated from the runs that exist.")
print()
if all_negative:
    print("The slope is negative in every seed, which is worth recording as an")
    print("observation. But position and task identity are confounded: all seeds share")
    print("one learning order, so the two tasks at the end are always the same two.")
    print("Running orders 02 and 03 would separate them. Until then the curve cannot")
    print("be attributed to plasticity rather than to task difficulty.")

## 7. Save

In [ ]:
import csv, datetime

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT = CACHE / "diagnostics" / f"stream_metrics_{ORDER}_{stamp}"
EXPORT.mkdir(parents=True, exist_ok=True)

record = {
    "notebook": "28_stream_metrics",
    "purpose": ("The adapter plasticity study's accuracy and forgetting metrics, computed "
                "for this framework so the two can be compared."),
    "comparability_notes": [
        "Task-incremental only. This framework has no single head over all classes.",
        "Eight tasks, not twenty. The per-10-task slope is an extrapolation.",
        "One learning order, so task identity and position are confounded.",
        "No within-seed repetitions, so the noise floor in the decision rule "
        "cannot be estimated and plasticity loss cannot be declared.",
    ],
    "code_revision": HEAD, "study_id": STUDY_ID, "order": ORDER,
    "order_tasks": ORDER_TASKS, "matrices": matrices, "result": result,
}
(EXPORT / "stream_metrics.json").write_text(json.dumps(record, indent=2) + "\n",
                                            encoding="utf-8")

with (EXPORT / "summary.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["metric", "mean", "sd", "ci_low", "ci_high"]
                    + [f"seed_{s}" for s in seeds])
    for key, label in labels.items():
        row = result["across_seeds"][key]
        writer.writerow([label, row["mean"], row["sd"], row["ci_low"], row["ci_high"]]
                        + row["values"])

with (EXPORT / "learning_curve.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["position", "task"] + [f"seed_{s}" for s in seeds] + ["mean"])
    for i, task in enumerate(ORDER_TASKS):
        values = [result["per_seed"][s]["diagonal"][i] for s in seeds]
        writer.writerow([i + 1, task] + values + [sum(values) / len(values)])

SAVE_VERIFIED = True
for name in ("stream_metrics.json", "summary.csv", "learning_curve.csv"):
    path = EXPORT / name
    try:
        text = path.read_text(encoding="utf-8")
        if name.endswith(".json"):
            json.loads(text)
        print(f"  verified  {name:<22} {path.stat().st_size:>8,} bytes")
    except Exception as problem:
        SAVE_VERIFIED = False
        print(f"  FAILED    {name:<22} {problem}")

print(f"\nSaved locally: {EXPORT}")
if SAVE_TO_R2 and SAVE_VERIFIED:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    for name in ("stream_metrics.json", "summary.csv", "learning_curve.csv"):
        store.upload(EXPORT / name, f"{destination}/{name}")
    print("Uploaded to R2:", destination)
elif SAVE_TO_R2:
    print("Not uploading: the local files did not verify.")

## 8. What is still missing

These five metrics are the part of the adapter study's measurement set that this framework
could supply from records it already had. The rest of its plasticity probe has not been run
here:

- **E1**, the random-label training loss, which measures raw capacity to change
- **E2 proper**: three unrelated probe tasks, several probe points, a learning-rate grid,
  and fork-and-discard. Notebook 26 is an approximation of E2 with one probe task from the
  same dataset family and two probe points
- **The head-only control** and `R_t`. This one may not transfer at all: the adapter study's
  control is a linear probe on frozen pretrained features, and this framework has no
  pretrained backbone, so its equivalent sits near chance and the ratio's denominator is
  close to zero
- **Feature effective rank**
- **The noise floor** from repeated probes at t = 0, without which the decision rule cannot
  be applied